# Bloque 2: Regularización y Estrategias de Modelamiento
## Tema 2: Regularización Implícita

**Autor:** Sofia Dextre | Hack with DSC PUCP  
**Nivel:** Avanzado  
**Duración estimada:** 45 minutos

---

### 🎯 Objetivo
Entender cómo controlar el sobreajuste **sin agregar una penalización a la función de pérdida**, sino cambiando cómo entrena el modelo (dónde se detiene, qué datos ve, con qué frecuencia se actualiza).

### 📚 Temas a cubrir
1. ¿Qué es regularización implícita vs explícita?
2. Early Stopping: cuándo detener el entrenamiento
3. Subsample: diversidad en observaciones
4. Colsample: diversidad en variables
5. La importancia del algoritmo de optimización
6. Comparación: Implícita vs Explícita

---

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ Librerías importadas")

## 1. Regularización Explícita vs Implícita

### Dos estrategias diferentes, mismo objetivo: reducir overfitting

#### 🔴 Regularización Explícita (Tema 1)
**Modifica la función objetivo:**
```
Objetivo = Loss + λ · Penalización(β)
```
- Agregamos un término de castigo a los coeficientes
- El modelo "siente" la penalización durante el entrenamiento
- Ejemplos: L1, L2, Elastic Net

#### 🟡 Regularización Implícita (Este tema)
**Modifica cómo aprende el modelo:**
```
Objetivo = Loss  (sin cambios)
Cambio: cuándo se detiene, qué datos ve, con qué frecuencia actualiza
```
- NO modificamos la función de pérdida
- Cambiamos el **proceso de entrenamiento**
- Ejemplos: Early Stopping, Subsample, Colsample, Dropout (en redes)

### Comparación visual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Explícita
ax = axes[0]
lambdas = np.logspace(-2, 1, 50)
train_loss = 1 / (1 + lambdas)  # Empeora con λ
reg_term = lambdas  # Penalización crece
total_loss = train_loss + 0.3 * reg_term  # Objetivo total

ax.plot(np.log10(lambdas), train_loss, 'o-', linewidth=2.5, markersize=5, label='Loss original', color='#2E86AB')
ax.plot(np.log10(lambdas), reg_term, 's-', linewidth=2.5, markersize=5, label='Penalización (λ·coef)', color='#FF6B6B')
ax.plot(np.log10(lambdas), total_loss, '^-', linewidth=2.5, markersize=5, label='Objetivo total', color='#51CF66', alpha=0.8)
ax.fill_between(np.log10(lambdas), train_loss, total_loss, alpha=0.2, color='#FF6B6B')
ax.set_xlabel('log₁₀(λ)', fontsize=12, fontweight='bold')
ax.set_ylabel('Valor', fontsize=12, fontweight='bold')
ax.set_title('Regularización EXPLÍCITA\n(L1, L2, Elastic Net)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

# Implícita
ax = axes[1]
iterations = np.arange(100)
train_curve = 1 - 0.005 * iterations  # Sigue mejorando
val_curve = 1 - 0.003 * iterations + 0.00002 * (iterations - 30)**2  # Empeora después de iter 30
val_curve = np.maximum(val_curve, 0)  # No negativo

ax.plot(iterations, train_curve, 'o-', linewidth=2.5, markersize=4, label='Train Loss', color='#2E86AB')
ax.plot(iterations, val_curve, 's-', linewidth=2.5, markersize=4, label='Validation Loss', color='#A23B72')
ax.axvline(30, color='green', linestyle='--', linewidth=2.5, alpha=0.7, label='Early Stopping (iter 30)')
ax.fill_between(iterations, val_curve, 1, alpha=0.2, color='#A23B72')
ax.fill_between(iterations, 0, val_curve, alpha=0.1, color='#2E86AB')
ax.set_xlabel('Iteración', fontsize=12, fontweight='bold')
ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
ax.set_title('Regularización IMPLÍCITA\n(Early Stopping, Subsample, etc.)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print("\n📊 Diferencias clave:")
print("  Explícita: El modelo 'siente' la penalización directamente en la función objetivo")
print("  Implícita: El modelo no sabe que está siendo regularizado; el entrenamiento lo hace naturalmente")

---
## 2. Early Stopping: "Saber cuándo parar"

### El concepto

**En modelos iterativos** (Boosting, Redes neuronales, etc.):
- El modelo aprende iteración tras iteración
- Al principio: Train y Validation mejoran juntos ✓
- En algún punto: Train sigue mejorando, Validation empeora → **OVERFITTING**
- **Early Stopping:** Detener cuando Validation deja de mejorar

### ¿Por qué funciona?

- Limita la **complejidad implícita** del modelo
- Un modelo entrenado menos es más simple que uno entrenado mucho
- Es como regularizar sin tocar la función de pérdida

### Práctica: Early Stopping en Gradient Boosting

In [ ]:
# Dataset
np.random.seed(42)
X, y = make_classification(
    n_samples=2000,
    n_features=50,
    n_informative=10,
    n_redundant=20,
    n_clusters_per_class=2,
    random_state=42
)

X = StandardScaler().fit_transform(X)

# Split: Train, Validation, Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

# Entrenar Gradient Boosting SIN early stopping
print("\n⏳ Entrenando modelo SIN early stopping...")
gb_no_stop = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    validation_fraction=0,  # Sin validación
    n_iter_no_change=None  # Sin early stopping
)
gb_no_stop.fit(X_train, y_train)

# Entrenar CON early stopping
print("⏳ Entrenando modelo CON early stopping...")
gb_with_stop = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    validation_fraction=0.2,  # Usar 20% para validación
    n_iter_no_change=10  # Parar si no mejora en 10 iteraciones
)
gb_with_stop.fit(X_train, y_train)

print(f"\n✓ Sin Early Stopping: {gb_no_stop.n_estimators} árboles")
print(f"✓ Con Early Stopping: {gb_with_stop.n_estimators_} árboles (se detuvo en iteración {gb_with_stop.n_estimators_})")

### Visualizar curvas de aprendizaje

In [ ]:
# Extraer scores por iteración (para visualizar)
# Gradient Boosting nos da staged_predict_proba

train_scores_no_stop = []
test_scores_no_stop = []

for pred in gb_no_stop.staged_predict_proba(X_train):
    train_scores_no_stop.append(roc_auc_score(y_train, pred[:, 1]))
    
for pred in gb_no_stop.staged_predict_proba(X_test):
    test_scores_no_stop.append(roc_auc_score(y_test, pred[:, 1]))

# Con early stopping (aprox.)
train_scores_with_stop = []
test_scores_with_stop = []

for pred in gb_with_stop.staged_predict_proba(X_train):
    train_scores_with_stop.append(roc_auc_score(y_train, pred[:, 1]))
    
for pred in gb_with_stop.staged_predict_proba(X_test):
    test_scores_with_stop.append(roc_auc_score(y_test, pred[:, 1]))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sin Early Stopping
ax = axes[0]
iterations_no_stop = range(len(train_scores_no_stop))
ax.plot(iterations_no_stop, train_scores_no_stop, 'o-', linewidth=2.5, markersize=4, label='Train', color='#2E86AB', alpha=0.8)
ax.plot(iterations_no_stop, test_scores_no_stop, 's-', linewidth=2.5, markersize=4, label='Test', color='#A23B72', alpha=0.8)
ax.fill_between(iterations_no_stop, train_scores_no_stop, test_scores_no_stop, alpha=0.2, color='#FF6B6B')
ax.set_xlabel('Iteración', fontsize=12, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=12, fontweight='bold')
ax.set_title('SIN Early Stopping (300 árboles)\n⚠️ Train sigue mejorando, Test empeora', fontsize=12, fontweight='bold', color='#FF6B6B')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.5, 1.0])

# Con Early Stopping
ax = axes[1]
iterations_with_stop = range(len(train_scores_with_stop))
ax.plot(iterations_with_stop, train_scores_with_stop, 'o-', linewidth=2.5, markersize=4, label='Train', color='#2E86AB', alpha=0.8)
ax.plot(iterations_with_stop, test_scores_with_stop, 's-', linewidth=2.5, markersize=4, label='Test', color='#51CF66', alpha=0.8)
ax.axvline(gb_with_stop.n_estimators_, color='green', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Parada (iter {gb_with_stop.n_estimators_})')
ax.fill_between(iterations_with_stop, train_scores_with_stop, test_scores_with_stop, alpha=0.2, color='#95E1D3')
ax.set_xlabel('Iteración', fontsize=12, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=12, fontweight='bold')
ax.set_title('CON Early Stopping\n✓ Se detiene cuando Test deja de mejorar', fontsize=12, fontweight='bold', color='#51CF66')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.5, 1.0])

plt.tight_layout()
plt.show()

# Comparación de resultados
print("\n" + "="*70)
print("RESULTADOS: Early Stopping vs Sin Early Stopping")
print("="*70)
print(f"\nSIN Early Stopping:")
print(f"  Árboles entrenados: {len(train_scores_no_stop)}")
print(f"  Train AUC: {train_scores_no_stop[-1]:.4f}")
print(f"  Test AUC:  {test_scores_no_stop[-1]:.4f}")
print(f"  Overfitting (gap): {train_scores_no_stop[-1] - test_scores_no_stop[-1]:.4f}")

print(f"\nCON Early Stopping:")
print(f"  Árboles entrenados: {len(train_scores_with_stop)}")
print(f"  Train AUC: {train_scores_with_stop[-1]:.4f}")
print(f"  Test AUC:  {test_scores_with_stop[-1]:.4f}")
print(f"  Overfitting (gap): {train_scores_with_stop[-1] - test_scores_with_stop[-1]:.4f}")

print(f"\n✨ Mejora:")
print(f"  Árboles guardados: {len(train_scores_no_stop) - len(train_scores_with_stop)} menos")
print(f"  Mejor generalización: {test_scores_with_stop[-1] - test_scores_no_stop[-1]:.4f} en Test AUC")
print(f"  Menor sobreajuste: {(train_scores_no_stop[-1] - test_scores_no_stop[-1]) - (train_scores_with_stop[-1] - test_scores_with_stop[-1]):.4f}")
print("="*70)

---
## 3. Subsample: "Diversidad en Observaciones"

### El concepto

En Gradient Boosting (y otros algoritmos iterativos), en cada iteración:
- **Sin Subsample:** Cada árbol ve TODOS los datos de entrenamiento
- **Con Subsample:** Cada árbol ve una **muestra aleatoria** (ej: 80% de los datos)

### ¿Por qué reduce overfitting?

✓ **Diversidad:** Cada árbol aprende un patrón ligeramente diferente  
✓ **Menor memorización:** No todos los datos en cada iteración = no puede memorizar tanto  
✓ **Ensemble implícito:** Combinación de muchas subpoblaciones  

### Parámetro
```
subsample = 0.8  # Usar 80% de observaciones en cada iteración
```

### Práctica: Impacto de Subsample

In [ ]:
# Entrenar modelos con diferentes subsample valores
subsample_values = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
results_subsample = []

print("⏳ Entrenando con diferentes subsample values...")
for ss in subsample_values:
    model = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        subsample=ss,  # El parámetro que variamos
        random_state=42,
        validation_fraction=0.2,
        n_iter_no_change=15
    )
    model.fit(X_train, y_train)
    
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    overfitting = train_auc - test_auc
    
    results_subsample.append({
        'subsample': ss,
        'train_auc': train_auc,
        'test_auc': test_auc,
        'overfitting': overfitting,
        'n_trees': model.n_estimators_
    })
    
    print(f"  subsample={ss:.1f}: Test AUC={test_auc:.4f}, Overfitting gap={overfitting:.4f}")

df_subsample = pd.DataFrame(results_subsample)

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# AUC por subsample
ax = axes[0]
x_pos = range(len(subsample_values))
width = 0.35
ax.bar([p - width/2 for p in x_pos], df_subsample['train_auc'], width, label='Train', color='#2E86AB', alpha=0.8)
ax.bar([p + width/2 for p in x_pos], df_subsample['test_auc'], width, label='Test', color='#A23B72', alpha=0.8)
ax.set_xlabel('Subsample', fontsize=12, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=12, fontweight='bold')
ax.set_title('Rendimiento vs Subsample', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{v:.1f}' for v in subsample_values])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0.5, 1.0])

# Sobreajuste (gap)
ax = axes[1]
colors = ['#FF6B6B' if x > 0.05 else '#51CF66' for x in df_subsample['overfitting']]
ax.plot(subsample_values, df_subsample['overfitting'], 'o-', linewidth=2.5, markersize=10, color='#FF6B6B')
ax.fill_between(subsample_values, df_subsample['overfitting'], alpha=0.3, color='#FF6B6B')
ax.set_xlabel('Subsample', fontsize=12, fontweight='bold')
ax.set_ylabel('Overfitting (Train - Test AUC)', fontsize=12, fontweight='bold')
ax.set_title('Subsample reduce Sobreajuste', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.invert_xaxis()  # Subsample baja de izq a derecha

# Árboles entrenados
ax = axes[2]
ax.bar(range(len(subsample_values)), df_subsample['n_trees'], color='#FFE66D', alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Subsample', fontsize=12, fontweight='bold')
ax.set_ylabel('# Árboles (con Early Stopping)', fontsize=12, fontweight='bold')
ax.set_title('Early Stopping + Subsample\nMás árboles = más diversidad', fontsize=13, fontweight='bold')
ax.set_xticks(range(len(subsample_values)))
ax.set_xticklabels([f'{v:.1f}' for v in subsample_values])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✨ Insight: subsample < 1.0 reduce overfitting al introducir diversidad")

---
## 4. Colsample: "Diversidad en Variables"

### El concepto

Similar a Subsample, pero en **variables** en lugar de observaciones:
- **Sin Colsample:** Cada árbol considera TODAS las variables
- **Con Colsample:** Cada árbol solo ve una **muestra de variables** (ej: 70% de las features)

### Variantes

```
colsample_bytree = 0.7    # Cada árbol ve 70% de variables
colsample_bylevel = 0.7   # En cada nivel del árbol, 70% de variables
colsample_bynode = 0.7    # En cada split, 70% de variables
```

### ¿Por qué reduce overfitting?

✓ **Fuerza exploración:** No puede usar siempre las mismas variables  
✓ **Robustez:** Descubre patrones con diferentes subconjuntos  
✓ **Menos correlación entre árboles:** Cada uno "ve" un aspecto distinto

### Práctica: Subsample vs Colsample

In [ ]:
# Entrenar modelos con diferentes combinaciones
configs = [
    {'subsample': 1.0, 'colsample': 1.0, 'name': 'Sin diversidad'},
    {'subsample': 0.8, 'colsample': 1.0, 'name': 'Solo Subsample'},
    {'subsample': 1.0, 'colsample': 0.7, 'name': 'Solo Colsample'},
    {'subsample': 0.8, 'colsample': 0.7, 'name': 'Ambos (diversidad)'},
]

results_diversity = []

print("⏳ Entrenando con diferentes configuraciones...")
for config in configs:
    model = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        subsample=config['subsample'],
        colsample_bytree=config['colsample'],
        random_state=42,
        validation_fraction=0.2,
        n_iter_no_change=15
    )
    model.fit(X_train, y_train)
    
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    test_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    overfitting = train_auc - test_auc
    
    results_diversity.append({
        'config': config['name'],
        'subsample': config['subsample'],
        'colsample': config['colsample'],
        'train_auc': train_auc,
        'test_auc': test_auc,
        'overfitting': overfitting
    })
    
    print(f"  {config['name']:25s}: Test AUC={test_auc:.4f}, Gap={overfitting:.4f}")

df_diversity = pd.DataFrame(results_diversity)

# Tabla comparativa
print("\n" + "="*80)
print("TABLA COMPARATIVA: Impacto de Diversidad")
print("="*80)
print(df_diversity.to_string(index=False))
print("="*80)

### Visualizar impacto

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs_names = df_diversity['config'].tolist()
x_pos = range(len(configs_names))

# Train AUC
ax = axes[0, 0]
ax.bar(x_pos, df_diversity['train_auc'], color='#2E86AB', alpha=0.8, edgecolor='black', linewidth=1.5)
for i, v in enumerate(df_diversity['train_auc']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=11, fontweight='bold')
ax.set_title('Train AUC', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(configs_names, rotation=15, ha='right', fontsize=9)
ax.set_ylim([0.8, 1.0])
ax.grid(True, alpha=0.3, axis='y')

# Test AUC
ax = axes[0, 1]
colors_test = ['#FF6B6B' if v < 0.90 else '#51CF66' for v in df_diversity['test_auc']]
ax.bar(x_pos, df_diversity['test_auc'], color=colors_test, alpha=0.8, edgecolor='black', linewidth=1.5)
for i, v in enumerate(df_diversity['test_auc']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=11, fontweight='bold')
ax.set_title('Test AUC (más alto = mejor generalización)', fontsize=12, fontweight='bold', color='#51CF66')
ax.set_xticks(x_pos)
ax.set_xticklabels(configs_names, rotation=15, ha='right', fontsize=9)
ax.set_ylim([0.8, 1.0])
ax.grid(True, alpha=0.3, axis='y')

# Overfitting gap
ax = axes[1, 0]
colors_gap = ['#FF6B6B' if v > 0.05 else '#51CF66' for v in df_diversity['overfitting']]
ax.bar(x_pos, df_diversity['overfitting'], color=colors_gap, alpha=0.8, edgecolor='black', linewidth=1.5)
for i, v in enumerate(df_diversity['overfitting']):
    ax.text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Train AUC - Test AUC', fontsize=11, fontweight='bold')
ax.set_title('Sobreajuste (más bajo = mejor)', fontsize=12, fontweight='bold', color='#51CF66')
ax.set_xticks(x_pos)
ax.set_xticklabels(configs_names, rotation=15, ha='right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

# Resumen: Diversidad vs Performance
ax = axes[1, 1]
diversities = ['Ninguna', 'Subsample', 'Colsample', 'Ambas']
colors_div = ['#FF6B6B', '#FFE66D', '#95E1D3', '#51CF66']
y_pos = range(len(diversities))

# Mostrar el efecto
ax.barh(y_pos, [1 - (o/0.15) for o in df_diversity['overfitting']], 
        color=colors_div, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(diversities, fontsize=11, fontweight='bold')
ax.set_xlabel('Calidad de Generalización (normalizado)', fontsize=11, fontweight='bold')
ax.set_title('Efecto Acumulativo de Diversidad', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n✨ Conclusión: Subsample + Colsample mejora Test AUC y reduce sobreajuste")

---
## 5. El Algoritmo de Optimización También Elige

### Un insight importante

**Problema:** Dado los mismos datos y la misma función de pérdida, hay **muchas soluciones** que obtienen el mismo error en training.

**Pregunta:** ¿Cuál elige el algoritmo de optimización?

**Respuesta:** Depende del algoritmo:
- **SGD:** Actualiza con gradientes pequeños y ruidosos → favorece soluciones "planas"
- **Adam:** Adapta la tasa de aprendizaje → soluciones diferentes a SGD
- **Momentum:** Acelera en direcciones consistentes → soluciones distintas

### Implicación para regularización

Aunque la función objetivo es igual, el **camino que toma el algoritmo** durante el entrenamiento regulariza implícitamente.

### Ilustración: Múltiples soluciones con mismo train error

In [ ]:
# Crear un problema donde múltiples soluciones tienen el mismo train error
from sklearn.linear_model import Ridge

# Dataset con multicolinealidad
np.random.seed(42)
n_samples, n_features = 100, 30
X_col, y_col = make_classification(n_samples=n_samples, n_features=n_features, 
                                    n_informative=5, n_redundant=15, random_state=42)
X_col = StandardScaler().fit_transform(X_col)
X_train_col, X_test_col, y_train_col, y_test_col = train_test_split(
    X_col, y_col, test_size=0.3, random_state=42
)

# Entrenar varias soluciones con regularización diferente
alphas = [0.001, 0.01, 0.1, 1, 10]
train_accs = []
test_accs = []
coef_norms = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_col, y_train_col)
    
    train_accs.append(model.score(X_train_col, y_train_col))
    test_accs.append(model.score(X_test_col, y_test_col))
    coef_norms.append(np.linalg.norm(model.coef_))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Soluciones con train error similar pero test error diferente
ax = axes[0]
ax.scatter(coef_norms, train_accs, s=300, alpha=0.6, c=range(len(alphas)), 
           cmap='Blues', edgecolor='black', linewidth=2, label='Train')
ax.scatter(coef_norms, test_accs, s=300, alpha=0.6, c=range(len(alphas)), 
           cmap='Reds', edgecolor='black', linewidth=2, marker='s', label='Test')

for i, alpha in enumerate(alphas):
    ax.annotate(f'α={alpha}', (coef_norms[i], train_accs[i]), 
               xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')

ax.set_xlabel('Norma de coeficientes ||β||', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Múltiples soluciones = Mismo Train, Diferentes Test', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Sobreajuste
ax = axes[1]
gaps = np.array(train_accs) - np.array(test_accs)
colors = ['#FF6B6B' if g > 0.05 else '#51CF66' for g in gaps]
ax.bar(range(len(alphas)), gaps, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_xticks(range(len(alphas)))
ax.set_xticklabels([f'{a}' for a in alphas])
ax.set_xlabel('λ (regularización)', fontsize=12, fontweight='bold')
ax.set_ylabel('Sobreajuste (Train - Test)', fontsize=12, fontweight='bold')
ax.set_title('Regularización controla la complejidad de la solución', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n🔍 Insight: Aunque train error es similar, las soluciones con ||β|| mayor generalizan peor")
print("    Esto es porque generalizan a patrones más complejos (sobreajuste)")

---
## 6. Comparación: Explícita vs Implícita

### Tabla resumen

In [ ]:
comparison_table = pd.DataFrame({
    'Aspecto': [
        'Qué modifica',
        'Cuándo usar',
        'Ventaja principal',
        'Desventaja',
        'Parámetros clave',
        'Interpretabilidad',
        'Complejidad de tuning'
    ],
    'EXPLÍCITA (L1, L2, Elastic Net)': [
        'Función objetivo (Loss + λ·Penalty)',
        'Modelos lineales, menos iteraciones',
        'Control directo, bien entendida',
        'Requiere elegir tipo (L1 vs L2)',
        'λ, α (mezcla)',
        'Muy alta (puedes ver los coeficientes)',
        'Media (pocas decisiones)'
    ],
    'IMPLÍCITA (ES, Subsample, Colsample)': [
        'Proceso de entrenamiento',
        'Modelos iterativos (Boosting, redes)',
        'Modelos más simples + poderosos',
        'Más parámetros para tuning',
        'n_iter, subsample, colsample, lr',
        'Media (depende del modelo)',
        'Alta (muchas decisiones)'
    ]
})

print("\n" + "="*100)
print("COMPARACIÓN: Regularización Explícita vs Implícita")
print("="*100)
print(comparison_table.to_string(index=False))
print("="*100)

### Decisión: ¿Cuál usar?

In [ ]:
decision_guide = {
    '🔴 Regresión Logística / Lineales': 'Explícita (L1, L2, Elastic Net)',
    '🟡 Gradient Boosting': 'Implícita (Early Stopping + Subsample + Colsample)',
    '🟡 Random Forest': 'Implícita (Subsample, Colsample) - Menos Early Stopping',
    '🔵 Redes Neuronales': 'Implícita (Early Stopping, Dropout) + Explícita (L2)',
    '🟢 SVM': 'Explícita (C es equivalente a 1/λ)',
}

print("\n" + "="*80)
print("GUÍA RÁPIDA: Qué regularización usar por algoritmo")
print("="*80)
for algo, reg in decision_guide.items():
    print(f"{algo:40s} → {reg}")
print("="*80)

print("\n✨ Mejor práctica: Usa AMBAS")
print("  • Regularización Explícita para controlar magnitud de coeficientes")
print("  • Regularización Implícita para controlar complejidad del proceso")

---
## 7. Caso Real: Fraud Detection

Apliquemos todo lo aprendido en un caso real.

In [ ]:
# Dataset de fraude
from sklearn.datasets import make_classification

X_fraud, y_fraud = make_classification(
    n_samples=5000,
    n_features=100,
    n_informative=20,
    n_redundant=40,
    weights=[0.95, 0.05],  # 95% no fraude, 5% fraude
    random_state=42
)

X_fraud = StandardScaler().fit_transform(X_fraud)
X_train_fr, X_test_fr, y_train_fr, y_test_fr = train_test_split(
    X_fraud, y_fraud, test_size=0.3, random_state=42
)

# Baseline: Sin regularización
print("Modelo 1: Sin regularización...")
gb_baseline = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=8,
    random_state=42
)
gb_baseline.fit(X_train_fr, y_train_fr)
baseline_auc = roc_auc_score(y_test_fr, gb_baseline.predict_proba(X_test_fr)[:, 1])

# Con regularización implícita
print("Modelo 2: Con regularización implícita...")
gb_implicit = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,  # Más lento
    max_depth=5,  # Más superficial
    subsample=0.8,  # 80% observaciones
    colsample_bytree=0.7,  # 70% variables
    validation_fraction=0.2,
    n_iter_no_change=20,  # Early stopping
    random_state=42
)
gb_implicit.fit(X_train_fr, y_train_fr)
implicit_auc = roc_auc_score(y_test_fr, gb_implicit.predict_proba(X_test_fr)[:, 1])

# Comparar
print(f"\n{'='*70}")
print("RESULTADOS: Fraud Detection")
print(f"{'='*70}")
print(f"\nBaseline (sin regularización):")
print(f"  Árboles: {gb_baseline.n_estimators}")
print(f"  Test AUC: {baseline_auc:.4f}")

print(f"\nCon Regularización Implícita:")
print(f"  Árboles: {gb_implicit.n_estimators_} (paró en Early Stopping)")
print(f"  Test AUC: {implicit_auc:.4f}")

print(f"\nMejora: {implicit_auc - baseline_auc:+.4f} en AUC")
print(f"{'='*70}")

---
## 📝 Resumen: Tabla de Decisión Rápida

### Cuándo usar Regularización Implícita

| Técnica | Cuándo | Parámetro | Efecto |
|---------|--------|-----------|--------|
| **Early Stopping** | Modelo iterativo (Boosting, RNN) | `n_iter_no_change=N` | Detiene cuando val deja de mejorar |
| **Subsample** | Reducir overfitting en Boosting | `subsample=0.7-0.9` | Diversidad en observaciones |
| **Colsample** | Datos con redundancia | `colsample=0.6-0.8` | Diversidad en variables |
| **Learning Rate** | Siempre | `learning_rate=0.01-0.1` | Más bajo = regulariza más |
| **Max Depth** | Árboles pequeños | `max_depth=3-5` | Limita complejidad |

### Mejor configuración para Gradient Boosting

```python
GradientBoostingClassifier(
    # Control de iteraciones
    n_estimators=500,  # Muchos, pero...
    validation_fraction=0.2,
    n_iter_no_change=20,  # ...párate aquí
    
    # Regularización implícita
    learning_rate=0.05,  # Lento = regulariza
    subsample=0.8,  # Diversidad en filas
    colsample_bytree=0.7,  # Diversidad en columnas
    max_depth=5,  # Árboles poco profundos
)
```

---

## ✅ Checkpoints de Aprendizaje

- [ ] Entiendo la diferencia entre regularización explícita e implícita
- [ ] Puedo explicar cómo Early Stopping reduce overfitting
- [ ] Sé por qué Subsample y Colsample introducen diversidad
- [ ] Entiendo que el algoritmo de optimización también regulariza
- [ ] Puedo configurar un Gradient Boosting con regularización implícita
- [ ] Sé cuándo usar cada técnica

---

**Próximo tema:** Ensambles, Stacking y Modelos Híbridos
